# Method 1 (KeyBERT + scispaCy)

### Load Models

In [1]:
import spacy
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer

# Load medical NER model
nlp = spacy.load("en_core_sci_md")

# Load sentence transformer for KeyBERT
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
kw_model = KeyBERT(model=sentence_model)

/home/maaz-rafiq/anaconda3/envs/CamelMedAgents-RAG/lib/python3.10/site-packages/torch/cuda/__init__.py:184: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
/home/maaz-rafiq/anaconda3/envs/CamelMedAgents-RAG/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1298.83it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+-

### Preprocessing Function

In [2]:
import re

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text

### Medical Entity Extraction (scispaCy)

In [3]:
def extract_medical_entities(text):
    doc = nlp(text)
    
    entities = []
    for ent in doc.ents:
        entities.append(ent.text.lower())
    
    return list(set(entities))

In [5]:
print(extract_medical_entities("I have severe chest pain and shortness of breath for 3 days"))

['days', 'chest pain', 'shortness of breath']


### Semantic Keyword Extraction (KeyBERT)

In [9]:
def extract_keywords_keybert(text, top_n=5):
    keywords = kw_model.extract_keywords(
        text,
        keyphrase_ngram_range=(1, 3),
        stop_words='english',
        use_maxsum=True,
        nr_candidates=20,
        top_n=top_n
    )
    
    return [kw[0] for kw in keywords]

In [10]:
print(extract_keywords_keybert("I have severe chest pain and shortness of breath for 3 days"))

['days', 'shortness', 'severe', 'breath', 'severe chest pain']


### Fusion Strategy

In [11]:
def fuse_keywords(entities, keywords):
    final_keywords = set()
    
    for e in entities:
        final_keywords.add(e)
        
    for k in keywords:
        final_keywords.add(k)
    
    return list(final_keywords)

### Complete Pipeline Function

In [12]:
def keyword_extraction_pipeline(text):
    
    text = preprocess(text)
    
    # Step 1: Medical entities
    medical_entities = extract_medical_entities(text)
    
    # Step 2: Semantic keywords
    semantic_keywords = extract_keywords_keybert(text)
    
    # Step 3: Fusion
    final_keywords = fuse_keywords(medical_entities, semantic_keywords)
    
    return {
        "medical_entities": medical_entities,
        "semantic_keywords": semantic_keywords,
        "final_keywords": final_keywords
    }

In [13]:
query = "I have fever, headache and vomiting for two days"

result = keyword_extraction_pipeline(query)

print(result)

{'medical_entities': ['vomiting', 'days', 'fever headache'], 'semantic_keywords': ['days', 'headache', 'vomiting days', 'fever', 'fever headache'], 'final_keywords': ['headache', 'vomiting', 'days', 'fever headache', 'vomiting days', 'fever']}
